# ML S3 · Notebook 05 — Is Model A Actually Better? And What the Errors Are Telling You

| | |
|---|---|
| **Session ID** | ML S3 · Notebook 05 of 05 |
| **Course position** | Machine Learning — Session 3 of 10 · **final notebook of the session** |
| **Block types** | 🧠 **Muscle block throughout — no AI assistance.** This is the session's payoff and it is built on statistics you already have. |
| **Prerequisites** | Notebook 04 (confusion matrix, precision/recall) · **your existing hypothesis-testing background** |
| **Connects back to** | **Notebook 04** — the two models that disagreed. **Your statistics course** — null hypotheses, p-values, confidence intervals. |
| **Connects forward to** | **ML S5** (model bake-off with statistical comparison) · **ML S6** (thresholds) · **ML S10** (CEP clinic) |
| **CEP linkage** | Employee Turnover requires comparing Logistic Regression, Random Forest and Gradient Boosting, then *justifying* the best. This notebook is how you justify it with evidence rather than assertion. |
| **Run requirements** | `pandas`, `numpy`, `scikit-learn`, `scipy`. Run `ML_S3_00_datasets.ipynb` first. |
| **Checkpoint file** | `subscription_churn.csv` |

## Learning Objectives

By the end of this notebook you will be able to:

1. **Explain why a difference in test scores is not evidence of a better model.**
2. **Build a bootstrap confidence interval** around any metric, from scratch.
3. **Run and interpret McNemar's test** for comparing two classifiers on the same test set.
4. **Explain why a paired test is the right tool**, connecting it to the paired tests you already know.
5. **Estimate how large a test set must be** to detect a difference of a given size.
6. **Perform structured error analysis** — categorise failures and derive the next action from the categories rather than from instinct.



## Table of Contents

| § | Section | Type |
|---|---|---|
| 1 | The question left open by Notebook 04 | Intuition |
| 2 | Your statistics background, restated for models | Bridge |
| 3 | Bootstrap confidence intervals | 🧠 Muscle |
| 4 | Comparing two models: why paired | Core |
| 5 | McNemar's test | 🧠 Muscle |
| 6 | How big a test set do you need? | Core |
| 7 | Error analysis: 30 errors, by hand | 🧠 Muscle |
| 8 | From categories to actions | Core |
| — | Common Pitfalls · FAQ · **Session Conclusion** | Wrap-up |

# 1. The question left open by Notebook 04

Notebook 04 finished with two models and an unresolved comparison. Let us rebuild them and look at the numbers again.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

churn = pd.read_csv('subscription_churn.csv')
X, y = churn.drop(columns='churned'), churn['churned']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

num_cols = ['tenure_months', 'monthly_charges', 'total_charges', 'num_support_tickets_6m',
            'avg_monthly_usage_gb', 'late_payments_12m', 'has_premium_support',
            'num_services', 'age', 'satisfaction_score']
cat_cols = ['contract_type', 'region']

logreg = Pipeline([
    ('prep', ColumnTransformer([('num', StandardScaler(), num_cols),
                                ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)])),
    ('model', LogisticRegression(max_iter=2000))]).fit(X_tr, y_tr)

rf = Pipeline([
    ('prep', ColumnTransformer([('num', 'passthrough', num_cols),
                                ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)])),
    ('model', RandomForestClassifier(n_estimators=250, min_samples_leaf=4,
                                     random_state=0, n_jobs=-1))]).fit(X_tr, y_tr)

lr_pred, rf_pred = logreg.predict(X_te), rf.predict(X_te)

print(f"{'':22s}{'accuracy':>10}{'recall':>10}{'precision':>11}{'F1':>9}")
for name, p in [('LogisticRegression', lr_pred), ('RandomForest', rf_pred)]:
    print(f"{name:22s}{accuracy_score(y_te,p):>10.4f}{recall_score(y_te,p):>10.4f}"
          f"{precision_score(y_te,p):>11.4f}{f1_score(y_te,p):>9.4f}")
print(f"\nTest set size: {len(y_te)} customers, of whom {y_te.sum()} churned.")

Random Forest is ahead on accuracy. **Is that a real difference, or did the 2,400 customers who happened to land in this test set simply suit one model slightly better?**

This is not a philosophical question. You would answer it differently depending on the stakes:

- If the difference is real, ship Random Forest.
- If it is noise, ship whichever is simpler, cheaper to serve, or easier to explain — which is usually Logistic Regression.

**You already know how to answer this.** It is a hypothesis test, and you have been doing hypothesis tests since your statistics course. The only new thing is recognising that a model comparison *is* one.

# 2. Your statistics background, restated for models

Every idea you need here, you already have. Only the labels change.

| What you learned in statistics | The same idea, here |
|---|---|
| A sample statistic is not the population parameter | A test-set score is not the model's true quality |
| Sampling variability | Score changes when the test set changes |
| Confidence interval | Plausible range for the model's true score |
| Null hypothesis | "These two models are equally good" |
| p-value | Probability of seeing a gap this large if they were equal |
| Paired vs unpaired test | Both models scored on **the same customers** → paired |

The one habit to carry over unchanged is this:

> **A point estimate without an interval is an overclaim.**

You would never report a survey result as "43% support the policy" with no margin of error. Reporting "our model is 86.6% accurate" with no interval is the same overclaim, and it is standard practice in bad data science.

Notebook 02 already gestured at this by asking you to report the standard deviation across CV folds. This notebook makes it rigorous.

# 3. Bootstrap confidence intervals

> 🧠 **Muscle block.**

You have one test set of 2,400 customers and one accuracy figure. To know how much that figure would wobble, you would ideally collect many more test sets of 2,400 customers each — which you cannot do.

The **bootstrap** is a trick that gets you most of the way there using only the data you have:

1. Draw 2,400 customers **from your test set, with replacement**. Some appear twice, some not at all.
2. Compute the metric on that resample.
3. Repeat a few thousand times.
4. The spread of those values approximates the spread you would have seen across genuinely new test sets.

The logic — that resampling your sample mimics sampling the population — takes a moment to accept. Its justification is that your test set is your best available picture of the population, so perturbing it in the way sampling would perturb it gives a reasonable picture of sampling variability.

> ### 🌱 Just-in-Time — why *with replacement* is essential
>
> Sampling 2,400 rows from 2,400 rows **without** replacement returns the same 2,400 rows every time. Nothing varies and you learn nothing.
>
> **With** replacement, each draw is independent, so each resample is a slightly different mixture — some customers counted twice, others absent. That variation is a stand-in for the variation you would get from a fresh sample of the population.
>
> On average each resample contains about 63% of the distinct original rows. That number falls out of the mathematics and is worth recognising if you meet it elsewhere (it is also where "out-of-bag" estimation in Random Forests comes from — ML S5).

### Worked Example 1 — Bootstrap, written out step by step

In [ ]:
rng = np.random.default_rng(42)

y_true = y_te.to_numpy()
n_rows = len(y_true)

# --- one single resample, so you can see what happens ---
idx = rng.integers(0, n_rows, size=n_rows)      # with replacement
print(f"Original test set size          : {n_rows}")
print(f"Resample size                   : {len(idx)}")
print(f"Distinct customers in resample  : {len(np.unique(idx))} "
      f"({len(np.unique(idx))/n_rows:.1%} of the original)")
print(f"\nAccuracy on the full test set   : {accuracy_score(y_true, rf_pred):.4f}")
print(f"Accuracy on this one resample   : {accuracy_score(y_true[idx], rf_pred[idx]):.4f}")

In [ ]:
def bootstrap_metric(y_true, y_pred, metric_fn, n_boot=2000, seed=0):
    # Resample the test set with replacement, recomputing the metric each time.
    rng = np.random.default_rng(seed)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    n = len(y_true)
    out = np.empty(n_boot)
    for b in range(n_boot):
        i = rng.integers(0, n, size=n)
        out[b] = metric_fn(y_true[i], y_pred[i])
    return out

boot_acc = bootstrap_metric(y_true, rf_pred, accuracy_score, n_boot=2000)

lo, hi = np.percentile(boot_acc, [2.5, 97.5])
print(f"RandomForest accuracy")
print(f"  point estimate : {accuracy_score(y_true, rf_pred):.4f}")
print(f"  95% CI         : [{lo:.4f}, {hi:.4f}]")
print(f"  interval width : {hi-lo:.4f}")

That interval is the honest version of the number. It says: *given this much test data, the true accuracy is plausibly anywhere in that range.*

### Worked Example 2 — Intervals for both models, on the metric that matters

In [ ]:
def ci(y_true, y_pred, fn, label, seed=0):
    b = bootstrap_metric(y_true, y_pred, fn, n_boot=2000, seed=seed)
    lo, hi = np.percentile(b, [2.5, 97.5])
    print(f"  {label:34s} {fn(y_true, y_pred):.4f}   95% CI [{lo:.4f}, {hi:.4f}]")
    return lo, hi

print("ACCURACY")
a_lr = ci(y_true, lr_pred, accuracy_score, "LogisticRegression")
a_rf = ci(y_true, rf_pred, accuracy_score, "RandomForest")

print("\nRECALL  (the metric Notebook 04 argued for)")
r_lr = ci(y_true, lr_pred, recall_score, "LogisticRegression")
r_rf = ci(y_true, rf_pred, recall_score, "RandomForest")

In [ ]:
print("Do the 95% intervals overlap?\n")
print(f"  accuracy : LR [{a_lr[0]:.4f}, {a_lr[1]:.4f}]   RF [{a_rf[0]:.4f}, {a_rf[1]:.4f}]"
      f"   -> {'OVERLAP' if a_lr[1] > a_rf[0] else 'no overlap'}")
print(f"  recall   : LR [{r_lr[0]:.4f}, {r_lr[1]:.4f}]   RF [{r_rf[0]:.4f}, {r_rf[1]:.4f}]"
      f"   -> {'OVERLAP' if r_lr[1] > r_rf[0] else 'no overlap'}")

> ### ⚠️ A trap worth stopping on
>
> Comparing two confidence intervals by eye is **not** a valid significance test, and it is a mistake that appears constantly in reports.
>
> - **Non-overlapping intervals** do imply a significant difference. That direction is safe.
> - **Overlapping intervals do not imply no difference.** Two intervals can overlap substantially while the difference is still statistically significant.
>
> The reason is that these intervals were computed **independently**, throwing away the fact that both models were evaluated on *the same customers*. That shared information is exactly what makes a comparison sensitive — and ignoring it makes the eyeball test conservative and unreliable.
>
> To compare two models properly you need a **paired** test. That is Section 4.

### ✅ Quick Check

1. Why does the bootstrap sample with replacement?
2. Your 95% CI for accuracy is [0.71, 0.94]. What should you do?
3. Can you bootstrap any metric?

<details>
<summary><b>Answers</b></summary>

1. Without replacement, resampling n rows from n rows returns the identical set every time and produces zero variation. Replacement makes each resample a different mixture, mimicking the variability of drawing a fresh sample.
2. Get more test data. That interval is far too wide to support any decision — the model could be mediocre or excellent and you cannot tell. Reporting the midpoint as if it were the answer would be seriously misleading.
3. Essentially yes — accuracy, recall, F1, AUC, MAE, R², or a custom business cost. That generality is the bootstrap's great virtue: it needs no formula for the metric's sampling distribution, only the ability to compute it.

</details>

---

# 4. Comparing two models: why paired

## Intuition

Both models were evaluated on **the same 2,400 customers**. That is not a coincidence to be ignored — it is the most useful fact available.

Think of the analogy you already know. To test whether a training programme improves performance you can either:

- **Unpaired:** measure one group before, a different group after. Differences between the two groups' composition add noise.
- **Paired:** measure the *same* people before and after. Person-to-person variation cancels out, so a smaller real effect becomes detectable.

Paired designs are more sensitive because they remove a source of variation.

The same applies here. Some customers are simply easy to classify and both models get them right; some are genuinely ambiguous and both get them wrong. **Those cases carry no information about which model is better.** They are the equivalent of person-to-person variation, and a paired test discards them.

What matters is the cases where the models **disagree**.

### Worked Example 3 — The 2×2 that actually matters

In [ ]:
lr_correct = (lr_pred == y_true)
rf_correct = (rf_pred == y_true)

both_right = int(( lr_correct &  rf_correct).sum())
lr_only    = int(( lr_correct & ~rf_correct).sum())
rf_only    = int((~lr_correct &  rf_correct).sum())
both_wrong = int((~lr_correct & ~rf_correct).sum())

pd.DataFrame(
    [[both_right, lr_only],
     [rf_only,    both_wrong]],
    index=['RF correct', 'RF wrong'],
    columns=['LR correct', 'LR wrong'],
)

In [ ]:
print(f"  Both correct    : {both_right:5d}   <- no information about which is better")
print(f"  Both wrong      : {both_wrong:5d}   <- no information about which is better")
print(f"  Only LR correct : {lr_only:5d}   <-- these are the")
print(f"  Only RF correct : {rf_only:5d}   <-- DISCORDANT pairs")
print(f"\n  Discordant total: {lr_only + rf_only}  "
      f"({(lr_only+rf_only)/len(y_true):.1%} of the test set)")

**Look at how much data the pairing throws away — and why that is the right thing to do.**

The great majority of customers land in the two "both agree" cells. On those, the models are indistinguishable. Including them in the comparison would only dilute the signal.

The comparison rests entirely on the discordant cases. And the question becomes beautifully simple:

> **Of the cases where exactly one model was right, was it Random Forest more often than chance would explain?**

That is a coin-flip question. Under the null hypothesis that the models are equally good, each discordant case is a fair coin — equally likely to fall either way.

### 🖼️ Image Slot — Why pairing concentrates the evidence

**What to show:** A 2x2 grid of the four agreement cells, with cell areas drawn roughly proportional to their counts: two very large grey cells on the diagonal labelled 'both right' and 'both wrong', marked 'no information'. Two small coloured cells off-diagonal labelled 'only LR right' and 'only RF right', marked 'ALL the evidence lives here'. An arrow from the two small cells points to a coin-flip icon with the caption 'under the null, a fair coin'.

**Why here:** The counter-intuitive part of McNemar's test is that it discards most of the data. Showing the discarded cells as visually huge and the informative cells as visually tiny makes the logic land immediately.

**Placement:** Right after Worked Example 3, before Section 5.

**Alt text:** Diagram showing that only the discordant cells of the agreement matrix carry evidence.

<!-- INSERT IMAGE: s3_05_mcnemar_pairing.png -->

---

# 5. McNemar's test

> 🧠 **Muscle block.**

## Intuition

**McNemar's test** formalises the coin-flip question.

- **Null hypothesis (H₀):** the two models are equally good. Among the discordant cases, each is equally likely to favour either model.
- **Under H₀:** if there are *d* discordant cases, the number favouring Random Forest follows a Binomial(*d*, 0.5) distribution.
- **The test:** how surprising is the split we actually observed?

That is it. It is a binomial test on the discordant pairs, and you have done binomial tests before.

### Worked Example 4 — Computing it from first principles

In [ ]:
from scipy.stats import binomtest

d = lr_only + rf_only
print(f"Discordant cases: {d}")
print(f"  favouring RandomForest       : {rf_only}")
print(f"  favouring LogisticRegression : {lr_only}")
print(f"\nUnder H0 we would expect about {d/2:.0f} each way.")

result = binomtest(rf_only, n=d, p=0.5, alternative='two-sided')
print(f"\nExact McNemar (binomial) test")
print(f"  p-value = {result.pvalue:.5f}")

In [ ]:
alpha = 0.05
if result.pvalue < alpha:
    print(f"p = {result.pvalue:.5f} < {alpha}  ->  REJECT H0.")
    print("The difference between these two models is larger than sampling noise")
    print("comfortably explains. It is a real difference on this task.")
else:
    print(f"p = {result.pvalue:.5f} >= {alpha}  ->  fail to reject H0.")
    print("We do not have enough evidence to call these models different.")
    print("Choose on other grounds: simplicity, latency, interpretability, cost.")

### Worked Example 5 — Confirming with statsmodels

In [ ]:
try:
    from statsmodels.stats.contingency_tables import mcnemar
    table = [[both_right, lr_only], [rf_only, both_wrong]]
    print(mcnemar(table, exact=True))
except ImportError:
    print("statsmodels not installed — the hand-computed binomial test above is the same thing.")
    print("(McNemar's exact test IS a binomial test on the discordant pairs.)")

> ### 🌱 Just-in-Time — exact versus chi-squared
>
> You will meet McNemar's test in two forms:
>
> - **Exact (binomial).** What we computed. Always valid.
> - **Chi-squared approximation.** $\chi^2 = (b-c)^2/(b+c)$, using the two discordant counts. Faster, and fine when the discordant total is large (a common rule of thumb is above 25).
>
> Use the exact version. It costs nothing at these sizes and it has no validity conditions to remember.

## What McNemar does *not* tell you

Three limitations worth being explicit about, because a p-value invites over-reading:

1. **It tests accuracy, not the metric you care about.** McNemar compares *correctness* on each case. Notebook 04 argued that recall matters more here. A model can win on McNemar and still be the wrong choice.
2. **It says nothing about effect size.** With a very large test set, a difference of 0.2 percentage points becomes significant — and remains irrelevant. **Always report the size of the difference alongside the p-value.**
3. **It is specific to this test set and this task.** It does not generalise to other data.

### Worked Example 6 — Significance and importance are different questions

In [ ]:
diff = accuracy_score(y_true, rf_pred) - accuracy_score(y_true, lr_pred)
print(f"Observed accuracy difference : {diff:+.4f}  ({diff*100:+.2f} percentage points)")
print(f"McNemar p-value              : {result.pvalue:.5f}")
print()
print(f"In practical terms, across all {len(y_true)} test customers that is a difference of")
print(f"about {abs(diff)*len(y_true):.0f} additional correct predictions.")
print()
print("Statistical significance answers: 'is it real?'")
print("Effect size answers:              'is it worth anything?'")
print("You need both. Report both.")

> ### 🔎 2026 Reality Check
>
> Almost no applied ML work reports uncertainty on model comparisons. Leaderboards, blog posts and internal reports routinely present "model A: 0.873, model B: 0.869" and declare a winner, when the difference is well inside the noise for the test set size involved.
>
> This matters commercially. Teams ship more complex, slower, harder-to-maintain models on the strength of differences that would not survive a re-split — paying real serving and maintenance costs for an improvement that was never there.
>
> It is also a strong interview signal. Being able to say *"the difference was about half a point, McNemar gave p = 0.02, so it is real but small — and given the added serving cost I would still ship the simpler model"* demonstrates statistical literacy **and** engineering judgement in one sentence.
>
> Your cohort has an advantage here: you already know hypothesis testing. Most people entering this field from a pure programming background do not, and it shows.

### 🟢 Try It Yourself

You compare two fraud models on a test set of 5,000 transactions.

- Model A correct, Model B wrong: **31** cases
- Model B correct, Model A wrong: **58** cases
- Both correct: 4,802 · Both wrong: 109

Run McNemar's test by hand (you may use `binomtest`). Then answer: should you ship Model B?

<details>
<summary><b>Show solution</b></summary>

```python
from scipy.stats import binomtest
r = binomtest(58, n=31+58, p=0.5, alternative='two-sided')
print(r.pvalue)      # approximately 0.0055
```

**The statistics.** There are 89 discordant cases. Under H₀ we would expect about 44.5 favouring each model; we observed 58 favouring B. The two-sided p-value is roughly **0.0055**, so at α = 0.05 we reject H₀. The difference is real.

**Effect size.** B is correct on 27 more transactions out of 5,000 — about **0.54 percentage points** of accuracy.

**Should you ship B? Not yet — and the statistics are not what decides it.**

1. **Accuracy is the wrong metric for fraud.** The class is heavily imbalanced, so almost all of those 4,802 agreements are ordinary legitimate transactions. What you need to know is how the two models compare **on the fraudulent cases specifically** — recall — and McNemar on overall correctness does not tell you.

2. **The errors are not equally costly.** A missed fraud costs the value of the transaction; a false positive costs a blocked customer. Compute expected cost, as in Notebook 04 §7.

3. **Then weigh the engineering cost.** If B is substantially more expensive to serve or harder to explain to a regulator, half a point of accuracy may not justify it.

**The point of the exercise:** the test told you the difference is real. It did not tell you the difference is *useful*. Those are separate questions and you must answer both.

</details>

---

# 6. How big a test set do you need?

## Intuition

The question usually arrives backwards. People run an experiment, find nothing, and conclude the models are equivalent — when in fact their test set was never large enough to detect a difference of the size they cared about.

**Decide up front:** *how small a difference would still matter to me?* Then check whether your test set can detect it.

This is statistical power, and it is the same reasoning as sizing a survey or a clinical trial.

### Worked Example 7 — Simulating detectability

In [ ]:
def detection_rate(n_test, true_diff, base_acc=0.85, n_sims=400, seed=0):
    # Fraction of simulated experiments that detect a real difference of `true_diff`.
    rng = np.random.default_rng(seed)
    detected = 0
    for _ in range(n_sims):
        a_ok = rng.random(n_test) < base_acc
        b_ok = rng.random(n_test) < (base_acc + true_diff)
        b_only = int((~a_ok & b_ok).sum())
        a_only = int((a_ok & ~b_ok).sum())
        if b_only + a_only == 0:
            continue
        if binomtest(b_only, b_only + a_only, 0.5).pvalue < 0.05:
            detected += 1
    return detected / n_sims

print("Probability of detecting a REAL 2 percentage-point improvement:\n")
print(f"  {'test set size':>15}  {'detection rate':>15}")
for n_test in [250, 500, 1000, 2500, 5000, 10000]:
    print(f"  {n_test:>15}  {detection_rate(n_test, 0.02):>15.0%}")

Read the column. With a small test set, a genuine two-point improvement is missed most of the time — you would run the experiment, see nothing, and wrongly conclude the models are equivalent.

**A negative result from an underpowered experiment is not evidence of no difference.** It is evidence of insufficient data.

### Worked Example 8 — Smaller differences need much more data

In [ ]:
print("Detection rate at a fixed test size of 2,000:\n")
print(f"  {'true difference':>17}  {'detection rate':>15}")
for d_true in [0.01, 0.02, 0.03, 0.05, 0.10]:
    print(f"  {d_true*100:>15.0f} pp  {detection_rate(2000, d_true):>15.0%}")

The relationship is steep: **halving the difference you want to detect requires roughly four times the data.**

That has a practical consequence worth internalising. If two models differ by half a percentage point, you would need a very large test set to establish it — and at that point you should ask whether a half-point difference is worth any engineering effort at all.

**Often the right conclusion is: they are close enough that the decision should be made on other grounds.** Simplicity, latency, interpretability, and maintenance cost are all legitimate tie-breakers, and they are frequently more valuable than a fraction of a point of accuracy.

### ✅ Quick Check

1. Your McNemar test gives p = 0.31. Are the models equivalent?
2. Why does detecting smaller differences need disproportionately more data?
3. You have 300 test rows. What can you realistically establish?

<details>
<summary><b>Answers</b></summary>

1. **No.** You failed to reject H₀, which is not the same as accepting it. Either the models are genuinely similar, or your test set was too small to detect the difference. Check your power (§6) before concluding anything.
2. Because the noise in your estimate shrinks with the square root of sample size, while the signal you are trying to detect shrinks linearly. To keep the signal-to-noise ratio constant while halving the signal, you need roughly four times the data.
3. Only large differences — on the order of 5–10 percentage points. Anything smaller is beyond your resolution. Report confidence intervals, be explicit that the comparison is underpowered, and resist declaring a winner on a small gap.

</details>

---

# 7. Error analysis: 30 errors, by hand

> 🧠 **Muscle block — no AI assistance. This is the single highest-value habit in the notebook.**

## Intuition

You now know *how much* your model is wrong and whether one model is genuinely better than another.

Neither tells you **what to do next.**

For that you have to look at the actual mistakes. Not the aggregate — the individual rows. This is called **error analysis**, and it is the clearest dividing line between someone who has followed tutorials and someone who has shipped models.

The instinct when a model underperforms is to try another algorithm or tune hyperparameters. That instinct is usually wrong, and it is expensive. Looking at thirty errors typically takes twenty minutes and tells you far more than a week of tuning.

### The procedure

1. Pull a sample of errors — 20 to 50 is plenty.
2. Look at each one and assign it a **category**.
3. **Count the categories.**
4. Attack the biggest category.

Step 3 is what makes this a method rather than browsing. Without counting you will fix whichever error you happened to look at last.

### Worked Example 9 — Pull 30 errors

In [ ]:
test = X_te.copy()
test['actual']    = y_true
test['predicted'] = rf_pred
test['prob']      = rf.predict_proba(X_te)[:, 1]

errors = test[test.actual != test.predicted].copy()
errors['error_type'] = np.where(errors.actual == 1, 'FALSE NEGATIVE (missed churner)',
                                                    'FALSE POSITIVE (false alarm)')

print(f"Total errors: {len(errors)} out of {len(test)}  ({len(errors)/len(test):.1%})")
print()
print(errors.error_type.value_counts().to_string())

In [ ]:
sample = errors.sample(30, random_state=7).sort_values('prob', ascending=False)

sample[['error_type', 'prob', 'contract_type', 'tenure_months', 'satisfaction_score',
        'num_support_tickets_6m', 'monthly_charges', 'late_payments_12m']].round(3)

### Worked Example 10 — Categorise them

Look at the table above before running the next cell. Genuinely look. Read a few rows and form a hypothesis about what these errors have in common.

Here is a category scheme for this problem. In real work you would invent the categories yourself after reading the first ten rows — that invention *is* the skill.

| Category | Definition | What it would imply |
|---|---|---|
| **Borderline** | Predicted probability between 0.35 and 0.65 | The model was genuinely uncertain. A threshold change would flip many of these. |
| **Confident and wrong** | Probability below 0.2 or above 0.8, but incorrect | The model was sure and mistaken — the most concerning category. |
| **Short-tenure churner** | Missed churner with tenure under 7 months | A specific subgroup the model handles badly. |
| **Contradictory signals** | Features point both ways (e.g. high satisfaction *and* many tickets) | Possibly genuinely unpredictable — or a missing feature. |
| **Plausibly unpredictable** | Nothing in the features suggests the outcome | Irreducible noise. No model fixes this. |

In [ ]:
def categorise(row):
    borderline = 0.35 <= row.prob <= 0.65
    confident  = row.prob < 0.20 or row.prob > 0.80
    if row.actual == 1 and row.tenure_months < 7 and row.contract_type == 'Month-to-month':
        return 'short-tenure churner'
    if confident:
        return 'confident and wrong'
    if borderline:
        return 'borderline'
    if row.satisfaction_score >= 7 and row.num_support_tickets_6m >= 3:
        return 'contradictory signals'
    return 'plausibly unpredictable'

sample['category'] = sample.apply(categorise, axis=1)

print("CATEGORY TALLY (30 sampled errors)\n")
tally = sample.category.value_counts()
for cat, count in tally.items():
    print(f"  {cat:26s} {count:3d}  {'#' * count}")

In [ ]:
# Cross-tabulate: which category produces which kind of error?
pd.crosstab(sample.category, sample.error_type)

### Worked Example 11 — Check whether the biggest category generalises

A tally of 30 is a hypothesis, not a finding. Test it against all the errors.

In [ ]:
errors['category'] = errors.apply(categorise, axis=1)

print("Category distribution across ALL errors:\n")
full_tally = errors.category.value_counts()
for cat, count in full_tally.items():
    print(f"  {cat:26s} {count:4d}  ({count/len(errors):5.1%})")

print("\nBase rates in the full test set, for comparison:")
short_tenure = test[(test.tenure_months < 7) & (test.contract_type == 'Month-to-month')]
print(f"  short-tenure month-to-month customers : {len(short_tenure)} "
      f"({len(short_tenure)/len(test):.1%} of the test set)")
print(f"  ...of whom the model got wrong        : "
      f"{(short_tenure.actual != short_tenure.predicted).sum()} "
      f"({(short_tenure.actual != short_tenure.predicted).mean():.1%} error rate)")
print(f"\n  Overall error rate                    : {(test.actual != test.predicted).mean():.1%}")

**Two separate findings have just come out of this, and they are worth distinguishing.**

**Finding 1 — from the tally.** The largest single category is *confident and wrong*: cases where the model was sure and mistaken. That is the most concerning kind of error, and it is the one the raw count points at.

**Finding 2 — from the base-rate comparison, which is sharper.** Short-tenure month-to-month customers are only a small slice of the test set, yet the model gets them wrong at roughly **twice its overall error rate**. Compare those two percentages in the output above.

Finding 2 would have been invisible in the tally, because the subgroup is small — it contributes relatively few errors in absolute terms while failing at a much higher rate. **This is exactly why the base-rate check is not optional.** A category can be large simply because the group is large; what makes a finding actionable is a group failing *disproportionately*.

Recall from Notebook 00 that the data was generated with a deliberate **threshold effect**: brand-new month-to-month customers churn at a sharply elevated rate, and the jump is abrupt rather than gradual. Nobody told the error analysis this. It surfaced from thirty rows and a base-rate comparison.

**That is what error analysis does.** It finds the structure in your failures, and structure is something you can act on — here, an explicit early-tenure feature or interaction term, which is a far more targeted fix than "try another algorithm."

---

# 8. From categories to actions

Each category implies a different next step. This mapping is the reason for counting.

| Category | What it means | Action |
|---|---|---|
| **Borderline** dominates | Model is uncertain near the boundary | **Move the threshold** (ML S6). Cheapest possible fix — no retraining. |
| **Confident and wrong** dominates | Model is confidently mistaken | Check for **label errors** first, then for a missing feature. Worrying if it persists. |
| **One subgroup** dominates | Systematic blind spot | Add a feature capturing it, add an interaction term, or train a separate model for that segment. |
| **Contradictory signals** dominate | Features genuinely conflict | Usually a **missing feature** — go and find data about what actually drives the outcome. |
| **Plausibly unpredictable** dominates | Irreducible noise | **Stop modelling.** You are at the ceiling. More tuning will not help; more *data of a different kind* might. |

That last row deserves emphasis. **Knowing when to stop is a professional skill.** If most of your errors are genuinely unpredictable from the features available, further tuning is not diligence — it is wasted effort, and often it produces overfitting dressed up as improvement.

### Worked Example 12 — What our tally recommends

In [ ]:
top = full_tally.index[0]
print(f"Largest error category: '{top}'  ({full_tally.iloc[0]/len(errors):.0%} of all errors)\n")

actions = {
    'borderline':              "Tune the decision threshold (ML S6). No retraining required.",
    'confident and wrong':     "Audit labels for these rows, then look for a missing feature.",
    'short-tenure churner':    "Add an explicit early-tenure feature or interaction term; consider a\n"
                               "  dedicated model for new month-to-month customers.",
    'contradictory signals':   "Go and find additional data - the current features genuinely conflict.",
    'plausibly unpredictable': "You are near the ceiling. Stop tuning; invest in better data instead.",
}
print("Recommended next action:")
print(f"  {actions[top]}")
print()
print("Note what this is NOT: 'try XGBoost' or 'tune more hyperparameters'.")
print("Error analysis points at the specific problem, not at a bigger hammer.")
print()
print("And do not stop at the largest bucket. The short-tenure subgroup above")
print("fails at roughly twice the overall error rate despite being small - that")
print("is a second, independent finding with its own fix.")

> ### 🔎 2026 Reality Check
>
> Error analysis is the most consistently under-practised skill in applied machine learning, and it has become *more* valuable as models have become easier to train, not less.
>
> When fitting a model was hard, modelling skill was the bottleneck. Now that a competent model is a few lines away — and an AI assistant will write those lines for you — the bottleneck has moved to **knowing what is actually wrong and what to do about it**. That work cannot be delegated, because it requires understanding the domain and the decision.
>
> This is also why this section is a muscle block. Asking an assistant to "analyse my model's errors" produces plausible generic prose. Sitting with thirty rows and forming your own categories produces a finding.
>
> In interviews, "walk me through how you'd improve this model" is answered weakly with a list of algorithms and strongly with: *"I'd pull thirty errors, categorise them, and let the biggest category tell me what to do."*

### 🟢 Try It Yourself

You run error analysis on a loan-default model and categorise 40 errors:

- Borderline (probability 0.4–0.6): **6**
- Confident and wrong: **4**
- Self-employed applicants: **22**
- Plausibly unpredictable: **8**

What is your next action, and what would you explicitly *not* do?

<details>
<summary><b>Show solution</b></summary>

**The finding.** Self-employed applicants account for 22 of 40 errors — over half. That is a systematic blind spot in one identifiable subgroup, not diffuse noise.

**First, check the base rate.** If self-employed applicants are 55% of your data, this is proportional and unremarkable. If they are 8% of your data and 55% of your errors, you have found something substantial. **Always compare against the base rate before acting** — this is the step Worked Example 11 demonstrated, and skipping it is how people chase phantom patterns.

**Assuming it is disproportionate, the likely cause:** your income features probably do not describe self-employed applicants well. Salaried applicants have stable, verifiable monthly income; self-employed income is variable, seasonal, and often documented differently. A single `annual_income` column captures the first group well and the second poorly.

**Actions, in order of cost:**

1. Add features that describe income *variability* — income volatility, months of history, number of income sources.
2. Add an explicit interaction between employment type and income features, so the model can treat the two groups differently.
3. If the subgroup is large enough, train a separate model for it.
4. Consider whether the label itself means the same thing for both groups.

**What you would explicitly NOT do:** try a different algorithm, tune hyperparameters, or add more trees. The problem is a **missing feature for a specific subgroup**. No amount of algorithmic sophistication invents information that is not in the data. Switching to a gradient booster might recover a fraction of a point and would leave the actual problem untouched.

**One more thing worth flagging:** a model that systematically underperforms on self-employed applicants is a potential **fairness and regulatory issue**, not merely an accuracy issue. In lending, differential performance across identifiable groups can carry legal consequences. Error analysis surfaced it; it now needs escalating rather than quietly patching.

</details>

# Common Pitfalls

| Pitfall | What happens | Fix |
|---|---|---|
| Declaring a winner from raw scores | You ship complexity for noise | McNemar, plus an effect size |
| Comparing two CIs by eye | Overlapping intervals can still be significantly different | Use a paired test |
| Reading "not significant" as "equivalent" | Underpowered experiment mistaken for a finding | Check power (§6) |
| Reporting p without effect size | A trivial difference looks important | Report both, always |
| Using an unpaired test | Throws away the shared test set; loses sensitivity | McNemar is paired |
| Tuning instead of looking | Weeks spent on the wrong problem | Read 30 errors first |
| Reading errors without counting | You fix whichever one you saw last | Categorise and tally |
| Skipping the base-rate check | You chase a pattern that is just group size | Compare subgroup error rate to overall |

# FAQ

**Q: How many bootstrap resamples?**
1,000 is usually enough for a 95% interval; 2,000–10,000 for stable tails. It is cheap — use 2,000 unless the metric is very slow to compute.

**Q: Can I use McNemar for three models?**
Not directly — pairwise tests across three models inflate the false-positive rate. Either apply a correction (Bonferroni is the simplest) or use Cochran's Q, which handles multiple classifiers at once. For your CEP's three models, pairwise McNemar with a Bonferroni correction is perfectly defensible and easy to explain.

**Q: What if models were evaluated on different test sets?**
Then you cannot pair, and you have lost most of your sensitivity. Use the same test set for every model — this is another reason to fix your splits with a seed at the start of a project.

**Q: How many errors should I examine?**
20–50. Beyond that you are usually re-confirming categories you have already found. If new categories are still appearing at 50, keep going.

**Q: What if my model has thousands of errors?**
Sample. Thirty random errors are representative enough to find the dominant category. You can also sample *within* a slice you suspect — but do the random sample first, or you will only confirm what you already believed.

**Q: Does this apply to regression?**
Yes, with a different definition of "error". Take the rows with the largest absolute or percentage error and categorise those. Notebook 03 §6.3 was a coarse version of exactly this, broken down by city tier.

# Conclusion — and the end of Session 3

## What this notebook established

A difference in scores is a **measurement**, not a **finding**. Turning one into the other takes two things you already had:

1. **An interval, not a point.** The bootstrap gives you one for any metric, with no formula required.
2. **A paired test.** Both models saw the same customers; McNemar uses that fact, and the comparison rests entirely on the cases where they disagree.

And once you know whether the difference is real, the question becomes what to do next — which no aggregate metric can answer. **Thirty errors, categorised and counted, will tell you more than a week of hyperparameter tuning.**

## What Session 3 established, across all five notebooks

This session is the spine of the course, and here is the whole argument in order:

| Notebook | The question it answered |
|---|---|
| **01** | Is my evaluation set-up honest? *(Four leaks. Correlation cannot detect them; only timing can.)* |
| **02** | Is my estimate stable? *(One split is one opinion. Pipeline makes correct preprocessing structural.)* |
| **03** | Am I measuring the right thing? *(Every metric encodes a judgement about which errors matter.)* |
| **04** | What are my errors made of? *(Two error directions, usually with very different costs.)* |
| **05** | Is the difference real, and what do I do about it? *(Paired test, then error analysis.)* |

Notice that **not one of those questions is about algorithms.** You will meet many models in Sessions 4 and 5, and every one of them will be evaluated with what you built today. A practitioner who knows five algorithms and evaluates them honestly is far more valuable than one who knows fifty and cannot tell whether any of them works.

## Directly ahead of you

- **ML S4 — Classification and the model family map.** New algorithms, evaluated with this session's discipline. Logistic regression properly this time, decision trees, and a fast tour of the rest.
- **ML S6** turns the threshold dial that Notebook 04 kept pointing at, and completes roughly half of the Employee Turnover project.
- **ML S10** walks both CEP briefs line by line. If Sessions 3 and 6 have landed, that clinic is mostly assembly.

## Your Employee Turnover checklist, from this session

- [ ] Stratified 80:20 split, `random_state=123` — as the brief specifies *(NB 01)*
- [ ] 5-fold cross-validation on all three models *(NB 02)*
- [ ] SMOTE applied **inside** the CV folds, never before the split *(NB 01 §8, NB 02 §5 — the trap)*
- [ ] Confusion matrix for each model *(NB 04)*
- [ ] A written recall-versus-precision justification, argued from **costs** *(NB 04 §7)*
- [ ] A statistical comparison, not just "GBM had the highest score" *(NB 05)*

**Next session:** ML S4 — Classification and the model family map.